# Interpreting Models

You can use Azure Machine Learning to interpret a model by using an *explainer* that quantifies the amount of influence each feature contributes to the predicted label. There are many common explainers, each suitable for different kinds of modeling algorithm; but the basic approach to using them is the same.

## Explain a Model

Let's start with a model that is trained outside of Azure Machine Learning - Run the cell below to train a decision tree classification model.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve

# load the diabetes dataset
print("Loading Data...")
data = pd.read_csv('data/diabetes.csv')

# Separate features and labels
features = ['Pregnancies','PlasmaGlucose','DiastolicBloodPressure','TricepsThickness','SerumInsulin','BMI','DiabetesPedigree','Age']
labels = ['not-diabetic', 'diabetic']
X, y = data[features].values, data['Diabetic'].values

# Split data into training set and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

# Train a decision tree model
print('Training a decision tree model')
model = DecisionTreeClassifier().fit(X_train, y_train)

# calculate accuracy
y_hat = model.predict(X_test)
acc = np.average(y_hat == y_test)
print('Accuracy:', acc)

# calculate AUC
y_scores = model.predict_proba(X_test)
auc = roc_auc_score(y_test,y_scores[:,1])
print('AUC: ' + str(auc))

print('Model trained.')

The training process generated some model evaluation metrics based on a hold-back validation dataset, so you have an idea of how accurately it predicts; but how do the features in the data influence the prediction?

### Install the Interpretability Library
To find out, first install the open-source `interpret-community` library. You can use this to interpret many typical kinds of model - purely locally, without any connection to Azure Machine Learning - even if they haven't been trained in an Azure ML job or registered in an Azure ML workspace.

> **Pip may warn about a `semver` conflict** with the `responsibleai` package. That package is not used here - the responsible AI dashboard is built by components running in the cloud, not by this kernel. The warning can be ignored.

In [ ]:
# The shap version must match interpret-community - we pin the highest supported one.
# If an older shap was already imported in this kernel, restart the notebook kernel.
%pip install -q --upgrade interpret-community "shap==0.46.0"

### Get an Explainer for our Model

Now that you have the library installed, let's get a suitable explainer for the model. There are many kinds of explainer. In this example you'll use a *Tabular Explainer*, which is a "black box" explainer that can be used to explain many kinds of model by invoking an appropriate [SHAP](https://github.com/slundberg/shap) model explainer.

In [ ]:
from interpret.ext.blackbox import TabularExplainer

# "features" and "classes" fields are optional
tab_explainer = TabularExplainer(model, 
                             X_train, 
                             features=features, 
                             classes=labels)
print(tab_explainer, "ready!")

### Get Global Feature Importance

The first thing to do is try to explain the model by evaluating the overall *feature importance* - in other words, quantifying the extent to which each feature influences the prediction based on the whole training dataset.

In [ ]:
# you can use the training data or the test data here
global_tab_explanation = tab_explainer.explain_global(X_train)

# Get the top features by importance
global_tab_feature_importance = global_tab_explanation.get_feature_importance_dict()
for feature, importance in global_tab_feature_importance.items():
    print(feature,":", importance)

The feature importance is ranked, with the most important feature listed first.

### Get Local Feature Importance

So you have an overall view, but what about explaining individual observations? Let's generate *local* explanations for individual predictions, quantifying the extent to which each feature influenced the decision to predict each of the possible label values. In this case, it's a binary model, so there are two possible labels (non-diabetic and diabetic); and you can quantify the influence of each feature for each of these label values for individual observations in a dataset. You'll just evaluate the first two cases in the test dataset.

In [ ]:
# Get the observations we want to explain (the first two)
X_explain = X_test[0:2]

# Get predictions
predictions = model.predict(X_explain)

# Get local explanations
local_tab_explanation = tab_explainer.explain_local(X_explain)

# Get feature names and importance for each possible label
local_tab_features = local_tab_explanation.get_ranked_local_names()
local_tab_importance = local_tab_explanation.get_ranked_local_values()

for l in range(len(local_tab_features)):
    print('Support for', labels[l])
    label = local_tab_features[l]
    for o in range(len(label)):
        print("\tObservation", o + 1)
        feature_list = label[o]
        total_support = 0
        for f in range(len(feature_list)):
            print("\t\t", feature_list[f], ':', local_tab_importance[l][o][f])
            total_support += local_tab_importance[l][o][f]
        print("\t\t ----------\n\t\t Total:", total_support, "Prediction:", labels[predictions[o]])



## A Responsible AI Dashboard for a Registered Model

As you've seen, explanations for a model trained outside Azure ML can be generated with `interpret-community` directly in the notebook.

To attach explanations to a model **registered in the workspace**, you use a **Responsible AI dashboard**. It offers more than feature importance alone: error analysis, counterfactual "what if" examples, and causal analysis.

The dashboard is produced by a pipeline job built from ready-made components. You'll create it with the **wizard in Azure Machine Learning studio** - by clicking, without writing code. The wizard runs exactly that pipeline underneath.

> **Why not from the SDK**: the dashboard components are published in Microsoft's `azureml` registry and are not available in every workspace - fetching them with `registry_client.components.get()` then fails with `Could not find component with name`. The studio wizard is unaffected.

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential=credential)

# The wizard needs a registered MLflow model and data in mltable format
model = ml_client.models.get(name="diabetes_model", label="latest")
data = ml_client.data.get(name="diabetes_mltable", label="latest")

print(f"Model: {model.name}, version {model.version}, type {model.type}")
print(f"Data:  {data.name}, version {data.version}, type {data.type}")

### Build the Dashboard in Studio

Open [Azure Machine Learning studio](https://ml.azure.com) and follow the steps below. The cell above printed the names and versions you'll need.

1. In the left menu select **Models**, then the **diabetes_model** model.
2. On the **Details** tab select **Create Responsible AI dashboard (preview)**.
3. **Training dataset**: choose **diabetes_mltable**. The wizard accepts only data in `mltable` format - which is exactly why that asset was created in [Lab 4B](labdocs/Lab04B.md).
4. **Test dataset**: choose the same asset. In a real project this would be a separate set, but here the point is to learn the tool.
5. **Modeling task**: choose **Classification**.
6. **Dashboard components**: choose the **Model debugging** profile - it covers error analysis, counterfactual examples and model explanations.
7. **Component parameters**: set **Target feature** to **Diabetic** and make sure **Generate explanations** is on.
8. **Experiment configuration**: give the dashboard a name, pick an experiment and the **aml-cluster** compute, then select **Create**.

The job takes some fifteen minutes. You can follow its progress on the experiment page.

### Review the Explanations

When the job finishes, go back to the **diabetes_model** model and open its **Responsible AI** tab. Select the dashboard you created, and in it:

1. Look at the **Aggregate feature importance** chart - the global feature importance.
2. Switch to **Individual feature importance** and pick a single data point to see what drove that one prediction.
3. Open the **Error analysis** section - it shows which subgroups of the data the model gets wrong most often.

> **Compare with the results from the start of this lab**: the ranking of features should be close to the one `interpret-community` produced in the notebook. It's the same method, just run in the cloud and attached to a registered model.

**More information**: read about the dashboard wizard in [Generate Responsible AI insights in the studio UI](https://learn.microsoft.com/azure/machine-learning/how-to-responsible-ai-insights-ui), about building it from the SDK in [Generate Responsible AI insights with YAML and Python](https://learn.microsoft.com/azure/machine-learning/how-to-responsible-ai-insights-sdk-cli), and about reading the dashboard in [Use the Responsible AI dashboard](https://learn.microsoft.com/azure/machine-learning/how-to-responsible-ai-dashboard).